# Million-parameter Humanoid optimization with MJX and ENNx

This notebook directly optimizes a roughly one-million-parameter BF16 JAX policy from scalar MuJoCo task rewards. MJX runs batched Humanoid simulation on the T4, while CUDA-resident ENNx generates, scores, and selects dense whole-model perturbations without gradients.

## 1. Prepare a T4 runtime

Select **Runtime > Change runtime type > T4 GPU**. The binary ENNx wheel is compiled for T4 `sm_75`; no Rust or CUDA compiler toolchain runs in this notebook.

In [ ]:
import os
import subprocess
import sys
from importlib import metadata
from pathlib import Path

os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["JAX_DEFAULT_MATMUL_PRECISION"] = "highest"
os.environ["MUJOCO_GL"] = "egl"
assert sys.version_info[:2] == (3, 12)

numpy_version = metadata.version("numpy")
scipy_version = metadata.version("scipy")
constraints = Path("/tmp/ennx-colab-constraints.txt")
constraints.write_text(
    f"numpy=={numpy_version}\nscipy=={scipy_version}\n", encoding="utf-8"
)
print(f"preserving numpy={numpy_version} scipy={scipy_version}")

CUDA_WHEEL = (
    "https://github.com/Kvutza/ennx/releases/download/v0.1.1/"
    "ennx-0.1.1%2Bcuda75-cp312-cp312-manylinux_2_28_x86_64.whl"
)
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--constraint",
        str(constraints),
        "mujoco==3.6.0",
        "mujoco-mjx==3.6.0",
        "mediapy>=1.2",
    ],
    check=True,
)
assert metadata.version("numpy") == numpy_version
assert metadata.version("scipy") == scipy_version
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--no-deps",
        CUDA_WHEEL,
    ],
    check=True,
)

In [ ]:
import json
import platform
from pathlib import Path
from urllib.request import urlretrieve

import jax
import jax.numpy as jnp
import mediapy as media
import mujoco
from mujoco import mjx
from jax.flatten_util import ravel_pytree

from ennx.experimental import Bf16Search

gpu_devices = [device for device in jax.devices() if device.platform == "gpu"]
runtime = {
    "python": platform.python_version(),
    "jax": jax.__version__,
    "mujoco": mujoco.__version__,
    "devices": [str(device) for device in jax.devices()],
}
print(json.dumps(runtime, indent=2))
assert gpu_devices, "JAX did not discover the T4"
GPU = gpu_devices[0]

## 2. Load the pinned Humanoid model

The MJCF model and its three visual includes are fetched from the MuJoCo Playground `v0.2.0` release. Simulation itself uses `mujoco.mjx` directly.

In [ ]:
MODEL_ROOT = Path("/content/mjx_humanoid")
MODEL_BASE = (
    "https://raw.githubusercontent.com/google-deepmind/"
    "mujoco_playground/v0.2.0/mujoco_playground/_src/"
    "dm_control_suite/xmls/"
)
MODEL_FILES = (
    "humanoid.xml",
    "common/skybox.xml",
    "common/visual.xml",
    "common/materials.xml",
)
for relative in MODEL_FILES:
    target = MODEL_ROOT / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    if not target.exists():
        urlretrieve(MODEL_BASE + relative, target)

SIM_DT = 0.005
SUBSTEPS = 5
CTRL_DT = SIM_DT * SUBSTEPS
mj_model = mujoco.MjModel.from_xml_path(str(MODEL_ROOT / "humanoid.xml"))
mj_model.opt.timestep = SIM_DT
mjx_model = mjx.put_model(mj_model, impl="jax")
data_seed = mjx.make_data(mj_model, impl="jax")
torso_id = mj_model.body("torso").id
print(f"nq={mj_model.nq} nv={mj_model.nv} actions={mj_model.nu}")

## 3. Define the MJX task

The policy observes root-relative position and velocity. Reward combines standing, upright posture, forward velocity, and a small control penalty. Episodes stop after a fall or a non-finite state.

In [ ]:
def observe(data):
    return jnp.concatenate((data.qpos[2:], data.qvel))


def reset_env(key):
    qpos_key, qvel_key = jax.random.split(key)
    qpos = jnp.asarray(mj_model.qpos0)
    qpos = qpos + 0.005 * jax.random.uniform(
        qpos_key, qpos.shape, minval=-1.0, maxval=1.0
    )
    qvel = 0.005 * jax.random.normal(qvel_key, (mj_model.nv,))
    data = data_seed.replace(qpos=qpos, qvel=qvel, ctrl=jnp.zeros(mj_model.nu))
    return mjx.forward(mjx_model, data)


def step_env(data, action):
    action = jnp.clip(action, -1.0, 1.0)
    old_x = data.qpos[0]

    def physics_step(carry, unused):
        del unused
        carry = carry.replace(ctrl=action)
        return mjx.step(mjx_model, carry), None

    data, _ = jax.lax.scan(physics_step, data, None, length=SUBSTEPS)
    height = data.xpos[torso_id, 2]
    upright = data.xmat[torso_id, 2, 2]
    standing = jnp.clip((height - 0.75) / 0.65, 0.0, 1.0)
    posture = jnp.clip((upright + 0.1) / 1.1, 0.0, 1.0)
    velocity = (data.qpos[0] - old_x) / CTRL_DT
    control = 0.002 * jnp.mean(jnp.square(action))
    reward = standing * posture * (1.0 + 0.25 * jnp.tanh(velocity)) - control
    finite = jnp.all(jnp.isfinite(data.qpos)) & jnp.all(jnp.isfinite(data.qvel))
    done = (~finite) | (height < 0.65) | (height > 2.25)
    return data, reward, done


sample_data = jax.jit(reset_env)(jax.random.PRNGKey(0))
OBS_SIZE = int(observe(sample_data).size)
ACTION_SIZE = int(mj_model.nu)
print(f"observation={OBS_SIZE} action={ACTION_SIZE} dt={CTRL_DT}")

## 4. Create the million-parameter JAX policy

The wide random features and small output initialization provide a stable near-zero-action center while exposing a genuinely high-dimensional search surface. ENNx receives no derivatives from this policy or the simulator.

In [ ]:
HIDDEN_SIZE = 2048
BOTTLENECK_SIZE = 416


def init_policy(key):
    keys = jax.random.split(key, 3)

    def make_layer(layer_key, inputs, outputs, scale=1.0):
        limit = scale * jnp.sqrt(6.0 / (inputs + outputs))
        weight = jax.random.uniform(
            layer_key, (inputs, outputs), minval=-limit, maxval=limit
        )
        return weight, jnp.zeros(outputs, dtype=jnp.float32)

    first = make_layer(keys[0], OBS_SIZE, HIDDEN_SIZE)
    second = make_layer(keys[1], HIDDEN_SIZE, BOTTLENECK_SIZE)
    output = make_layer(keys[2], BOTTLENECK_SIZE, ACTION_SIZE, scale=0.05)
    return first + second + output


def apply_policy(params, obs):
    w1, b1, w2, b2, w3, b3 = params
    hidden = jax.nn.tanh(obs @ w1 + b1)
    hidden = jax.nn.tanh(hidden @ w2 + b2)
    return jax.nn.tanh(hidden @ w3 + b3)


float_params = tuple(
    jax.device_put(value, GPU) for value in init_policy(jax.random.PRNGKey(7))
)
PARAMETER_COUNT = sum(int(value.size) for value in float_params)
assert 900_000 <= PARAMETER_COUNT <= 1_000_000
print(f"policy_parameters={PARAMETER_COUNT:,}")

## 5. Compile batched task evaluation

Each policy is evaluated on independently randomized initial conditions. The repetition and round seeds make the natural reset noise exactly reproducible, while the returned mean reward remains on a stable scale for ENNx posterior scoring.

In [ ]:
import tomllib

CONFIG_TOML = """
[experiment]
repetitions = 10
rounds = 32
seed = 41

[search]
history_capacity = 8
batch_arms = 4
candidates = 8
neighbors = 8
acquisition = "thompson"

[evaluation]
environments = 8
rollout_steps = 256

[output]
root = "/content/ennx_mjx_runs"
"""
CONFIG = tomllib.loads(CONFIG_TOML)
REPETITIONS = int(
    os.environ.get("ENNX_REPETITIONS", CONFIG["experiment"]["repetitions"])
)
ROUNDS = int(os.environ.get("ENNX_ROUNDS", CONFIG["experiment"]["rounds"]))
BASE_SEED = int(os.environ.get("ENNX_SEED", CONFIG["experiment"]["seed"]))
HISTORY_CAPACITY = CONFIG["search"]["history_capacity"]
BATCH_ARMS = CONFIG["search"]["batch_arms"]
CANDIDATES = CONFIG["search"]["candidates"]
NEIGHBORS = CONFIG["search"]["neighbors"]
ACQUISITION = CONFIG["search"]["acquisition"]
EVAL_ENVS = CONFIG["evaluation"]["environments"]
ROLLOUT_STEPS = CONFIG["evaluation"]["rollout_steps"]
RESULT_ROOT = Path(os.environ.get("ENNX_OUTPUT", CONFIG["output"]["root"]))
print(CONFIG_TOML)


def eval_keys(seed, index, arms=1):
    key = jax.random.fold_in(jax.random.PRNGKey(seed), index)
    keys = jax.random.split(key, arms * EVAL_ENVS)
    if arms == 1:
        return keys
    return keys.reshape((arms, EVAL_ENVS) + keys.shape[1:])


@jax.jit
def score_policy(params, keys):
    data = jax.vmap(reset_env)(keys)
    active = jnp.ones(EVAL_ENVS, dtype=jnp.float32)
    totals = jnp.zeros(EVAL_ENVS, dtype=jnp.float32)

    def rollout_step(carry, unused):
        del unused
        data, active, totals = carry
        obs = jax.vmap(observe)(data)
        action = apply_policy(params, obs)
        data, reward, done = jax.vmap(step_env)(data, action)
        totals = totals + active * reward
        active = active * (~done).astype(jnp.float32)
        return (data, active, totals), None

    (_, _, totals), _ = jax.lax.scan(
        rollout_step, (data, active, totals), None, length=ROLLOUT_STEPS
    )
    values = totals / ROLLOUT_STEPS
    return jnp.mean(values), jnp.std(values)


@jax.jit
def score_batch(params, keys):
    return jax.vmap(score_policy)(params, keys)


float_reward, float_std = score_policy(float_params, eval_keys(BASE_SEED, 0))
float_reward.block_until_ready()
print(f"float_reward={float(float_reward):.5f} std={float(float_std):.5f}")

## 6. Flatten the policy into resident BF16 leaves

JAX flattens and reconstructs the policy PyTree. ENNx keeps one contiguous BF16 row on CUDA, while per-tensor perturbation scales and normalized metric weights prevent the two large matrices from erasing the smaller output layer.

In [ ]:
flat_params, unravel = ravel_pytree(float_params)
base_row = jax.device_put(flat_params.astype(jnp.bfloat16), GPU)
base_row.block_until_ready()


def decode_params(row):
    return unravel(row.astype(jnp.float32))


def decode_batch(rows):
    return jax.vmap(decode_params)(rows)


def device_batch(search, trials):
    rows = [jax.dlpack.from_dlpack(view) for view in search.rows(trials)]
    batch = jnp.stack(rows)
    batch.block_until_ready()
    del rows
    return batch


blocks = []
offset = 0
for key, value in enumerate(jax.tree.leaves(float_params), start=1):
    length = int(value.size)
    deviation = float(jnp.std(value))
    scale = max(0.05 * deviation, 1.0e-4)
    blocks.append((key, offset, length, scale, 1.0 / length))
    offset += length
assert offset == PARAMETER_COUNT
base_params = decode_params(base_row)
base_reward, base_std = score_policy(base_params, eval_keys(BASE_SEED, 0))
base_reward.block_until_ready()
warm_rows = jnp.broadcast_to(base_row, (BATCH_ARMS, base_row.size))
warm_keys = eval_keys(BASE_SEED, 1, BATCH_ARMS)
warm_reward, _ = score_batch(decode_batch(warm_rows), warm_keys)
warm_reward.block_until_ready()
del warm_rows, warm_keys, warm_reward
print(
    f"resident_bytes={base_row.nbytes:,} blocks={len(blocks)} "
    f"bf16_reward={float(base_reward):.5f} std={float(base_std):.5f}"
)

## 7. Optimize the policy with ENNx

Rust TuRBO owns trust-region adaptation, acceptance, and restart. Each repetition starts from the same BF16 policy, then uses independent perturbation and MJX reset streams. Repetitions run sequentially so ten-seed statistics do not multiply resident T4 memory. The default TOML runs the real 10-by-32 experiment; set `ENNX_REPETITIONS=1` and `ENNX_ROUNDS=2` for a quick T4 validation pass. Every round records Yubo-style proposal, evaluation, tell, total-time, environment-step, and incumbent fields.

In [ ]:
import gc
import random
import time


def run_seed(repetition):
    experiment_seed = BASE_SEED + repetition
    start_reward, start_std = score_policy(base_params, eval_keys(experiment_seed, 0))
    start_reward.block_until_ready()
    search = Bf16Search(
        base_row,
        float(start_reward),
        blocks,
        HISTORY_CAPACITY,
        max_pending=BATCH_ARMS,
        base_variance=float(start_std) ** 2 / EVAL_ENVS,
    )
    search.profile(True)
    seed_rng = random.Random(experiment_seed)
    best_reward = float(start_reward)
    best_std = float(start_std)
    best_row = jnp.array(base_row, copy=True)
    run_records = []
    y_curve = [best_reward]
    env_steps = EVAL_ENVS * ROLLOUT_STEPS
    run_start = time.perf_counter()

    for round_index in range(ROUNDS):
        round_start = time.perf_counter()
        seeds = [
            [seed_rng.getrandbits(64) for _ in range(CANDIDATES)]
            for _ in range(BATCH_ARMS)
        ]
        neighbors = min(NEIGHBORS, search.history_len)

        proposal_start = time.perf_counter()
        trials = search.ask_batch(
            seeds,
            neighbors,
            acquisition=ACQUISITION,
            seed=experiment_seed * ROUNDS + round_index,
        )
        candidate_rows = device_batch(search, trials)
        proposal_dt = time.perf_counter() - proposal_start
        score_ms, pick_ms, write_ms, kernel_ms = search.last_profile

        eval_start = time.perf_counter()
        keys = eval_keys(experiment_seed, round_index + 1, BATCH_ARMS)
        reward, reward_std = score_batch(decode_batch(candidate_rows), keys)
        reward.block_until_ready()
        variances = jnp.square(reward_std).astype(jnp.float32) / EVAL_ENVS
        eval_dt = time.perf_counter() - eval_start

        tell_start = time.perf_counter()
        accepted = search.tell_batch(trials, reward, variances)
        tell_dt = time.perf_counter() - tell_start
        rewards = [float(value) for value in jax.device_get(reward)]
        reward_stds = [float(value) for value in jax.device_get(reward_std)]
        best_index = max(range(BATCH_ARMS), key=rewards.__getitem__)
        if search.best > best_reward:
            best_reward = search.best
            best_std = reward_stds[best_index]
            best_row = jnp.array(candidate_rows[best_index], copy=True)
            best_row.block_until_ready()
        del candidate_rows, keys, reward, reward_std, variances

        env_steps_iter = BATCH_ARMS * EVAL_ENVS * ROLLOUT_STEPS
        env_steps += env_steps_iter
        round_dt = time.perf_counter() - round_start
        elapsed = time.perf_counter() - run_start
        y_curve.append(best_reward)
        run_records.append(
            {
                "repetition": repetition,
                "seed": experiment_seed,
                "round": round_index,
                "elapsed": elapsed,
                "proposal_dt": proposal_dt,
                "eval_dt": eval_dt,
                "tell_dt": tell_dt,
                "round_dt": round_dt,
                "env_steps_iter": env_steps_iter,
                "env_steps_total": env_steps,
                "y_best": best_reward,
                "ret_eval": max(rewards),
                "ret_mean": sum(rewards) / len(rewards),
                "trust_length": search.length,
                "accepted": sum(accepted),
                "score_ms": score_ms,
                "pick_ms": pick_ms,
                "write_ms": write_ms,
                "kernel_ms": kernel_ms,
            }
        )
        print(
            f"ITER: seed = {experiment_seed} iter = {round_index} "
            f"elapsed = {elapsed:.2f}s eval_dt = {eval_dt:.3f}s "
            f"proposal_dt = {proposal_dt:.3f}s tell_dt = {tell_dt:.3f}s "
            f"env_steps_iter = {env_steps_iter} env_steps_total = {env_steps} "
            f"y_best = {best_reward:.5f} ret_eval = {max(rewards):.5f}"
        )
    return {
        "records": run_records,
        "curve": y_curve,
        "best_reward": best_reward,
        "best_std": best_std,
        "best_row": best_row,
    }


RESULT_ROOT.mkdir(parents=True, exist_ok=True)
(RESULT_ROOT / "config.toml").write_text(CONFIG_TOML, encoding="utf-8")
records = []
curves = []
best_reward = -float("inf")
best_std = float("nan")
best_row = jnp.array(base_row, copy=True)
trace_path = RESULT_ROOT / "trace.jsonl"
with trace_path.open("w", encoding="utf-8") as handle:
    for repetition in range(REPETITIONS):
        result = run_seed(repetition)
        records.extend(result["records"])
        curves.append(result["curve"])
        for record in result["records"]:
            handle.write(json.dumps(record, sort_keys=True) + "\n")
        handle.flush()
        if result["best_reward"] > best_reward:
            best_reward = result["best_reward"]
            best_std = result["best_std"]
            best_row = result["best_row"]
        del result
        gc.collect()

In [ ]:
import matplotlib.pyplot as plt


def mean_sem(values):
    array = jnp.asarray(values, dtype=jnp.float32)
    mean = jnp.mean(array, axis=0)
    if array.shape[0] == 1:
        sem = jnp.zeros_like(mean)
    else:
        sem = jnp.std(array, axis=0, ddof=1) / jnp.sqrt(array.shape[0])
    return (
        [float(value) for value in jax.device_get(mean)],
        [float(value) for value in jax.device_get(sem)],
    )


def metric_rows(name, scale=1.0):
    return [
        [
            scale * record[name]
            for record in records
            if record["repetition"] == repetition
        ]
        for repetition in range(REPETITIONS)
    ]


def plot_band(axis, x_values, values, label):
    mean, sem = mean_sem(values)
    lower = [value - error for value, error in zip(mean, sem)]
    upper = [value + error for value, error in zip(mean, sem)]
    axis.plot(x_values, mean, linewidth=2, label=label)
    axis.fill_between(x_values, lower, upper, alpha=0.2)
    return mean, sem


base_steps = EVAL_ENVS * ROLLOUT_STEPS
round_steps = BATCH_ARMS * EVAL_ENVS * ROLLOUT_STEPS
curve_steps = [base_steps + index * round_steps for index in range(ROUNDS + 1)]
rounds = list(range(1, ROUNDS + 1))
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
y_mean, y_sem = plot_band(axes[0], curve_steps, curves, "y_best +- SEM")
axes[0].set(xlabel="control transitions", ylabel="best mean reward per step")
axes[0].legend()

plot_band(axes[1], rounds, metric_rows("proposal_dt", 1_000), "ask")
plot_band(axes[1], rounds, metric_rows("eval_dt", 1_000), "MJX")
plot_band(axes[1], rounds, metric_rows("tell_dt", 1_000), "tell")
plot_band(axes[1], rounds, metric_rows("round_dt", 1_000), "complete round")
axes[1].set(xlabel="round", ylabel="milliseconds, mean +- SEM")
axes[1].set_yscale("log")
axes[1].legend()

profile_names = ["score", "pick", "write"]
profile_values = [
    sum(record[f"{name}_ms"] for record in records) / len(records)
    for name in profile_names
]
axes[2].bar(profile_names, profile_values, color=["#3d6f8e", "#cf8a3a", "#528a5b"])
axes[2].set(ylabel="milliseconds", title="CUDA proposal mean")
fig.tight_layout()
FIGURE_PATH = RESULT_ROOT / "curves.png"
fig.savefig(FIGURE_PATH, dpi=160, bbox_inches="tight")

summary = {
    "runtime": runtime,
    "parameters": PARAMETER_COUNT,
    "repetitions": REPETITIONS,
    "rounds": ROUNDS,
    "final_y_best_mean": y_mean[-1],
    "final_y_best_sem": y_sem[-1],
    "global_y_best": best_reward,
    "control_transitions": curve_steps[-1],
    "proposal_ms_mean": 1_000
    * sum(record["proposal_dt"] for record in records)
    / len(records),
    "evaluation_ms_mean": 1_000
    * sum(record["eval_dt"] for record in records)
    / len(records),
    "tell_ms_mean": 1_000 * sum(record["tell_dt"] for record in records) / len(records),
    "round_ms_mean": 1_000
    * sum(record["round_dt"] for record in records)
    / len(records),
}
(RESULT_ROOT / "summary.json").write_text(
    json.dumps(summary, indent=2, sort_keys=True), encoding="utf-8"
)
print(json.dumps(summary, indent=2))
print(f"artifacts={RESULT_ROOT}")

## 8. Render the optimized Humanoid

The final rollout is simulated with the selected BF16 policy. MuJoCo renders the stored MJX states from the tracking camera and writes an MP4 alongside the inline player.

In [ ]:
def run_rollout(params, steps=512):
    reset = jax.jit(reset_env)
    advance = jax.jit(step_env)
    act = jax.jit(apply_policy)
    data = reset(jax.random.PRNGKey(101))
    trajectory = []
    total = 0.0
    for _ in range(steps):
        trajectory.append(
            tuple(jax.device_get(value) for value in (data.qpos, data.qvel, data.ctrl))
        )
        action = act(params, observe(data))
        data, reward, done = advance(data, action)
        total += float(reward)
        if bool(done):
            break
    return trajectory, total


def render_video(trajectory, path):
    render_every = 2
    renderer = mujoco.Renderer(mj_model, height=480, width=640)
    cpu_data = mujoco.MjData(mj_model)
    frames = []
    for qpos, qvel, ctrl in trajectory[::render_every]:
        cpu_data.qpos[:] = qpos
        cpu_data.qvel[:] = qvel
        cpu_data.ctrl[:] = ctrl
        mujoco.mj_forward(mj_model, cpu_data)
        renderer.update_scene(cpu_data, camera="side")
        frames.append(renderer.render())
    renderer.close()
    fps = 1.0 / (CTRL_DT * render_every)
    media.write_video(path, frames, fps=fps)
    return frames, fps


best_params = decode_params(best_row)
trajectory, video_reward = run_rollout(best_params)
VIDEO_PATH = RESULT_ROOT / "humanoid.mp4"
frames, fps = render_video(trajectory, str(VIDEO_PATH))
print(
    f"frames={len(frames)} rollout_reward={video_reward:.3f} "
    f"best_eval={best_reward:.5f} +- {best_std:.5f} path={VIDEO_PATH}"
)
media.show_video(frames, fps=fps, loop=False)

## 9. What this measures

This is a direct high-dimensional black-box control experiment. ENNx searches approximately 972,000 BF16 policy parameters using only repeated noisy scalar task rewards. Candidate generation, distance scoring, acquisition, selection, full-row materialization, acceptance, history, and TuRBO adaptation execute in CUDA-Oxide on the T4. JAX receives selected policies and returns reward statistics through DLPack. Python only orchestrates repetitions, records small summaries, plots `y_best +- SEM`, and renders the global incumbent.